# EVE Frontier Solar System Viewer

Interactive notebook for visualizing a single solar system in detail.

This notebook provides:
- Solar system search and selection
- Interactive 3D visualization of all celestial objects
- Detailed statistics and object information
- Customizable visualization parameters

## Setup and Configuration

Import required libraries and configure the environment.

In [ ]:
# Standard library imports
import os
import sys
from pathlib import Path

# Add project root to path for imports
project_root = Path('/workspace')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Third-party imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Project imports
from evefrontier_datasets.data_loader import load_eve_data
from evefrontier_datasets.data_analysis import (
    search_solar_systems,
    get_solar_system_info,
    find_system_with_most_planets,
)
from evefrontier_datasets.visualization import visualize_solar_system

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

print("✓ All imports successful")
print(f"✓ Project root: {project_root}")

## Load EVE Frontier Data

Load the static data from the EVE Frontier datasets.

In [ ]:
# Load data from the latest release
# Set force_download=True to re-download even if file exists
db_data = load_eve_data(release_tag='e6c3', force_download=False)

print(f"\n✅ Data loaded successfully!")
print(f"   Tables available: {len(db_data)}")

## Browse Available Solar Systems

View a list of all solar systems in the database.

In [ ]:
# Display solar systems with their statistics
if 'SolarSystems' in db_data:
    systems_df = db_data['SolarSystems']
    
    print(f"\n🌍 Solar Systems Database")
    print("=" * 100)
    print(f"Total Systems: {len(systems_df):,}\n")
    
    # Create a summary with statistics
    system_stats = []
    
    for _, row in systems_df.iterrows():
        system_id = row['solarSystemId']
        system_name = row['name']
        info = get_solar_system_info(db_data, system_id)
        
        if info:
            system_stats.append({
                'System ID': system_id,
                'System Name': system_name,
                'Planets': info['planet_count'],
                'Moons': info['moon_count'],
                'Stations': info['station_count'],
                'Total Objects': info['planet_count'] + info['moon_count'] + info['station_count'] + 1
            })
    
    # Create DataFrame and sort by total objects
    stats_df = pd.DataFrame(system_stats)
    stats_df = stats_df.sort_values('Total Objects', ascending=False)
    
    print("\nTop 20 Systems by Total Objects:")
    print("-" * 100)
    print(stats_df.head(20).to_string(index=False))
    
    print("\n\nAll Systems Summary:")
    print("-" * 100)
    print(stats_df.to_string(index=False))
    
else:
    print("✗ SolarSystems table not found")

## Search for a Specific Solar System

Search for solar systems by name. Modify the `search_query` variable to find different systems.

In [ ]:
# Search for solar systems by name
search_query = 'Brana'  # 🔧 CHANGE THIS to search for different systems

print(f"\n🔎 Searching for systems matching: '{search_query}'")
print("=" * 100)

results = search_solar_systems(db_data, search_query)

if len(results) > 0:
    print(f"\n✓ Found {len(results)} matching system(s):\n")
    
    for idx, (_, row) in enumerate(results.iterrows(), 1):
        system_id = row['solarSystemId']
        system_name = row['name']
        
        # Get detailed info
        info = get_solar_system_info(db_data, system_id)
        if info:
            print(f"{idx}. {system_name}")
            print(f"   System ID: {system_id}")
            print(f"   Planets: {info['planet_count']}")
            print(f"   Moons: {info['moon_count']}")
            print(f"   Stations: {info['station_count']}")
            print(f"   Total Objects: {info['planet_count'] + info['moon_count'] + info['station_count'] + 1}")
            print()
else:
    print(f"\n✗ No systems found matching '{search_query}'")
    print("\n💡 Tip: Try a partial name or check the list above for available systems")

## Select System to Visualize

Choose a solar system to visualize. You can either:
1. Use the search results from above (set `use_search_result = True`)
2. Specify a system ID directly (set `use_search_result = False` and provide `system_id`)
3. Use the system with the most planets (set `use_most_planets = True`)

In [ ]:
# Configuration for system selection
use_search_result = True     # 🔧 Use the first result from search above
use_most_planets = False     # 🔧 Use system with most planets
system_id = None             # 🔧 Or specify a system ID directly (e.g., 30000142)

# Select the system
selected_system_id = None
selected_system_name = None

if use_most_planets:
    print("\n🌍 Finding system with most planets...")
    best_system = find_system_with_most_planets(db_data)
    if best_system:
        selected_system_id = best_system['system_id']
        selected_system_name = best_system['name']
        print(f"✓ Selected: {selected_system_name} (ID: {selected_system_id})")
        print(f"  Planets: {best_system['planet_count']}, Moons: {best_system['moon_count']}, Stations: {best_system['station_count']}")

elif use_search_result and len(results) > 0:
    selected_row = results.iloc[0]
    selected_system_id = selected_row['solarSystemId']
    selected_system_name = selected_row['name']
    print(f"\n✓ Selected from search: {selected_system_name} (ID: {selected_system_id})")

elif system_id is not None:
    selected_system_id = system_id
    if 'SolarSystems' in db_data:
        systems_df = db_data['SolarSystems']
        system_info = systems_df[systems_df['solarSystemId'] == system_id]
        if len(system_info) > 0:
            selected_system_name = system_info.iloc[0]['name']
            print(f"\n✓ Selected by ID: {selected_system_name} (ID: {selected_system_id})")
        else:
            print(f"\n✗ System ID {system_id} not found")
            selected_system_id = None
else:
    print("\n⚠️  No system selected")
    print("   Please either:")
    print("   1. Run the search cell above and set use_search_result = True")
    print("   2. Set use_most_planets = True")
    print("   3. Set a specific system_id value")

if selected_system_id:
    # Get detailed info
    info = get_solar_system_info(db_data, selected_system_id)
    if info:
        print(f"\n📊 System Details:")
        print(f"   Name: {selected_system_name}")
        print(f"   ID: {selected_system_id}")
        print(f"   Planets: {info['planet_count']}")
        print(f"   Moons: {info['moon_count']}")
        print(f"   Stations: {info['station_count']}")
        print(f"   Total Objects: {info['planet_count'] + info['moon_count'] + info['station_count'] + 1}")

## Visualize the Solar System

Generate an interactive 3D visualization of the selected solar system.

**Visualization Features:**
- **Color Coding:**
  - 🔴 Red: Star (sun at center)
  - 🔵 Blue: Planets
  - 🟢 Green: Moons
  - 🟠 Orange: NPC Stations

- **Interactive Controls:**
  - Click and drag to rotate
  - Scroll to zoom
  - Hover over objects for details
  - Toggle object types on/off in the legend

- **Scaling:**
  - Uses logarithmic scaling to handle huge coordinate ranges
  - All coordinates are relative to the star (center)
  - Size represents object type, not physical size

In [ ]:
# Visualization parameters
size_scale = 1.0  # �� Adjust to make markers larger (>1.0) or smaller (<1.0)

if selected_system_id:
    print(f"\n⭐ Visualizing Solar System: {selected_system_name}")
    print("=" * 100)
    
    system_plot_data = visualize_solar_system(
        db_data,
        selected_system_id,
        selected_system_name,
        size_scale=size_scale
    )
    
    if system_plot_data is not None:
        print(f"\n✅ Visualization complete for {selected_system_name}")
else:
    print("\n⚠️  No system selected for visualization")
    print("   Run the selection cell above first")

## Object Details and Statistics

View detailed information about all objects in the solar system.

In [ ]:
if selected_system_id and system_plot_data is not None:
    print(f"\n📋 Detailed Object List for {selected_system_name}")
    print("=" * 100)
    
    # Display by type
    for obj_type in ['Star', 'Planet', 'Moon', 'NPC Station']:
        type_objects = system_plot_data[system_plot_data['type'] == obj_type]
        if len(type_objects) > 0:
            print(f"\n{obj_type}s ({len(type_objects)}):")
            print("-" * 100)
            
            for idx, obj in type_objects.iterrows():
                # Calculate distance from star
                dist = np.sqrt(obj['x_centered']**2 + obj['y_centered']**2 + obj['z_centered']**2)
                
                print(f"  • {obj['name']}")
                print(f"    Position (relative): X={obj['x_centered']:.2e}, Y={obj['y_centered']:.2e}, Z={obj['z_centered']:.2e}")
                print(f"    Distance from star: {dist:.2e}")
                
    # Summary statistics
    print(f"\n\n📊 Coordinate Statistics")
    print("=" * 100)
    
    # Exclude star for distance calculations
    objects_only = system_plot_data[system_plot_data['type'] != 'Star']
    
    if len(objects_only) > 0:
        # Calculate distances
        distances = np.sqrt(
            objects_only['x_centered']**2 + 
            objects_only['y_centered']**2 + 
            objects_only['z_centered']**2
        )
        
        print(f"\nDistance from Star:")
        print(f"  Minimum: {distances.min():.2e}")
        print(f"  Maximum: {distances.max():.2e}")
        print(f"  Mean: {distances.mean():.2e}")
        print(f"  Median: {distances.median():.2e}")
        
        print(f"\nCoordinate Ranges (relative to star):")
        print(f"  X: [{objects_only['x_centered'].min():.2e}, {objects_only['x_centered'].max():.2e}]")
        print(f"  Y: [{objects_only['y_centered'].min():.2e}, {objects_only['y_centered'].max():.2e}]")
        print(f"  Z: [{objects_only['z_centered'].min():.2e}, {objects_only['z_centered'].max():.2e}]")
        
        # Type distribution
        print(f"\nObject Type Distribution:")
        type_counts = system_plot_data['type'].value_counts()
        for obj_type, count in type_counts.items():
            print(f"  {obj_type}: {count}")
    
else:
    print("\n⚠️  No plot data available")
    print("   Run the visualization cell above first")

## Distance Distribution Analysis

Analyze the distribution of object distances from the star.

In [ ]:
if selected_system_id and system_plot_data is not None:
    # Exclude star for distance calculations
    objects_only = system_plot_data[system_plot_data['type'] != 'Star']
    
    if len(objects_only) > 0:
        # Calculate distances
        distances = np.sqrt(
            objects_only['x_centered']**2 + 
            objects_only['y_centered']**2 + 
            objects_only['z_centered']**2
        )
        
        objects_with_dist = objects_only.copy()
        objects_with_dist['distance'] = distances
        
        print(f"\n📏 Distance Distribution for {selected_system_name}")
        print("=" * 100)
        
        # Create histogram
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        # Linear scale histogram
        ax1.hist(distances, bins=30, edgecolor='black', alpha=0.7)
        ax1.set_xlabel('Distance from Star', fontsize=12)
        ax1.set_ylabel('Number of Objects', fontsize=12)
        ax1.set_title(f'{selected_system_name} - Distance Distribution (Linear)', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        
        # Log scale histogram
        ax2.hist(distances, bins=30, edgecolor='black', alpha=0.7)
        ax2.set_xlabel('Distance from Star', fontsize=12)
        ax2.set_ylabel('Number of Objects', fontsize=12)
        ax2.set_title(f'{selected_system_name} - Distance Distribution (Log Scale)', fontsize=14, fontweight='bold')
        ax2.set_xscale('log')
        ax2.grid(True, alpha=0.3, which='both')
        
        plt.tight_layout()
        plt.show()
        
        # Distance by object type
        print(f"\n📊 Distance Statistics by Object Type:")
        print("-" * 100)
        
        for obj_type in objects_with_dist['type'].unique():
            type_data = objects_with_dist[objects_with_dist['type'] == obj_type]
            type_distances = type_data['distance']
            
            print(f"\n{obj_type}:")
            print(f"  Count: {len(type_distances)}")
            print(f"  Min Distance: {type_distances.min():.2e}")
            print(f"  Max Distance: {type_distances.max():.2e}")
            print(f"  Mean Distance: {type_distances.mean():.2e}")
            print(f"  Median Distance: {type_distances.median():.2e}")
        
        print(f"\n✓ Analysis complete")
    
else:
    print("\n⚠️  No plot data available")
    print("   Run the visualization cell above first")

## Export System Data

Export the solar system data to CSV for further analysis.

In [ ]:
# Export configuration
export_data = False  # 🔧 Set to True to export data

if export_data and selected_system_id and system_plot_data is not None:
    # Create export directory
    export_dir = Path('/workspace/data/exports')
    export_dir.mkdir(parents=True, exist_ok=True)
    
    # Generate filename
    safe_name = selected_system_name.replace(' ', '_').replace('/', '_')
    export_file = export_dir / f"solar_system_{safe_name}_{selected_system_id}.csv"
    
    # Export data
    system_plot_data.to_csv(export_file, index=False)
    
    print(f"\n✅ Data exported successfully!")
    print(f"   File: {export_file}")
    print(f"   Rows: {len(system_plot_data)}")
    print(f"   Columns: {list(system_plot_data.columns)}")
    
elif export_data:
    print("\n⚠️  No data to export")
    print("   Run the visualization cell above first")
else:
    print("\n💡 To export data, set export_data = True and run this cell again")

## Quick System Switcher

Quickly switch to a different system and visualize it.

Just run this cell with a new system name or ID to generate a complete visualization.

In [ ]:
# Quick visualization - change these values
quick_search = None      # 🔧 System name to search for (e.g., 'Brana')
quick_system_id = None   # 🔧 Or specify system ID directly
quick_size_scale = 1.0   # 🔧 Marker size scale

if quick_search or quick_system_id:
    quick_system_id_to_use = None
    quick_system_name = None
    
    # Search by name
    if quick_search:
        quick_results = search_solar_systems(db_data, quick_search)
        if len(quick_results) > 0:
            quick_system_id_to_use = quick_results.iloc[0]['solarSystemId']
            quick_system_name = quick_results.iloc[0]['name']
            print(f"✓ Found: {quick_system_name} (ID: {quick_system_id_to_use})")
        else:
            print(f"✗ No system found matching '{quick_search}'")
    
    # Use direct ID
    elif quick_system_id:
        quick_system_id_to_use = quick_system_id
        if 'SolarSystems' in db_data:
            systems_df = db_data['SolarSystems']
            system_info = systems_df[systems_df['solarSystemId'] == quick_system_id]
            if len(system_info) > 0:
                quick_system_name = system_info.iloc[0]['name']
    
    # Visualize if found
    if quick_system_id_to_use:
        print(f"\n⭐ Quick Visualization: {quick_system_name}")
        print("=" * 100)
        
        # Get info
        info = get_solar_system_info(db_data, quick_system_id_to_use)
        if info:
            print(f"\nSystem Details:")
            print(f"  Planets: {info['planet_count']}")
            print(f"  Moons: {info['moon_count']}")
            print(f"  Stations: {info['station_count']}")
            print(f"  Total Objects: {info['planet_count'] + info['moon_count'] + info['station_count'] + 1}")
        
        # Visualize
        quick_plot_data = visualize_solar_system(
            db_data,
            quick_system_id_to_use,
            quick_system_name,
            size_scale=quick_size_scale
        )
        
        if quick_plot_data is not None:
            print(f"\n✅ Quick visualization complete")
else:
    print("\n💡 Set quick_search or quick_system_id to visualize a system")
    print("   Example: quick_search = 'Brana'")
    print("   Example: quick_system_id = 30000142")

## Conclusion

This notebook provides a focused interface for exploring individual solar systems in EVE Frontier.

**Key Features:**
- Browse all available solar systems
- Search systems by name
- Interactive 3D visualization with color-coded object types
- Detailed statistics and object information
- Distance distribution analysis
- Data export capabilities
- Quick system switcher for rapid exploration

**Tips:**
- Use the Quick System Switcher cell for rapid exploration
- Adjust `size_scale` to change marker sizes
- Toggle object types in the visualization legend
- Export data for external analysis
- Combine with `spatial_exploration.ipynb` for broader spatial analysis

**Next Steps:**
- Explore different solar systems
- Compare systems by planet/moon counts
- Analyze spatial distribution patterns
- Create custom visualizations with modified parameters